In [1]:
import geopandas as gpd

building_path = r"data/Buildings/A020102.shp"
buildings_raw = gpd.read_file(building_path)

buildings_raw.head()


,STRATO,TEMA,CLASSE,ID_ZRIL,FEATURE_ID,EDIFC_TY,EDIFC_USO,EDIFC_STAT,EDIFC_ID,EDIFC_CASS,geometry
0,02,01,02,MI_2006_0033,000000000001,0101,0291,0401,E00000000001,000000001,"POLYGON Z ((517456.068 5038120.2 124.339, 5174..."
1,02,01,02,MI_2006_0033,000000000002,0101,0201,0403,E00000000002,000000002,"POLYGON Z ((518363.144 5037839.01 123.396, 518..."
2,02,01,02,MI_2006_0033,000000000003,0101,0201,0403,E00000000003,000000003,"POLYGON Z ((517244.845 5038001.943 124.193, 51..."
3,02,01,02,MI_2006_0033,000000000004,0101,0201,0403,E00000000004,000000004,"POLYGON Z ((517265.035 5038016.095 124.231, 51..."
4,02,01,02,MI_2006_0033,000000000005,0101,0201,0403,E00000000005,000000005,"POLYGON Z ((517248.421 5038020.853 124.398, 51..."


In [2]:
print("Number of buildings:", len(buildings_raw))
print("CRS:", buildings_raw.crs)
print("Columns:", buildings_raw.columns)


Number of buildings: 53041
CRS: EPSG:6707
Columns: Index(['STRATO', 'TEMA', 'CLASSE', 'ID_ZRIL', 'FEATURE_ID', 'EDIFC_TY',
       'EDIFC_USO', 'EDIFC_STAT', 'EDIFC_ID', 'EDIFC_CASS', 'geometry'],
      dtype='object')


In [3]:
# 1) Reproject from EPSG:6707 to EPSG:4326 (WGS84 lat/lon)
buildings = buildings_raw.to_crs(epsg=4326)

print(buildings.crs)
buildings.head()


EPSG:4326


,STRATO,TEMA,CLASSE,ID_ZRIL,FEATURE_ID,EDIFC_TY,EDIFC_USO,EDIFC_STAT,EDIFC_ID,EDIFC_CASS,geometry
0,02,01,02,MI_2006_0033,000000000001,0101,0291,0401,E00000000001,000000001,"POLYGON Z ((9.22342 45.49639 124.3394, 9.22342..."
1,02,01,02,MI_2006_0033,000000000002,0101,0201,0403,E00000000002,000000002,"POLYGON Z ((9.23502 45.49384 123.3962, 9.23492..."
2,02,01,02,MI_2006_0033,000000000003,0101,0201,0403,E00000000003,000000003,"POLYGON Z ((9.22071 45.49534 124.1928, 9.22097..."
3,02,01,02,MI_2006_0033,000000000004,0101,0201,0403,E00000000004,000000004,"POLYGON Z ((9.22097 45.49546 124.2312, 9.22097..."
4,02,01,02,MI_2006_0033,000000000005,0101,0201,0403,E00000000005,000000005,"POLYGON Z ((9.22076 45.49551 124.3979, 9.22076..."


In [4]:
buildings = buildings.rename(columns={
    "EDIFC_ID": "building_id",
    "EDIFC_TY": "building_type",
    "EDIFC_USO": "building_use",
    "EDIFC_STAT": "building_status"
})

buildings.columns


Index(['STRATO', 'TEMA', 'CLASSE', 'ID_ZRIL', 'FEATURE_ID', 'building_type',
       'building_use', 'building_status', 'building_id', 'EDIFC_CASS',
       'geometry'],
      dtype='object')

In [5]:
# Final structure check
print(buildings.head())
print(buildings.columns)
print("CRS:", buildings.crs)

# Save cleaned version with all attributes preserved
output_path = r"data/Buildings/buildings_clean_full.geojson"
buildings.to_file(output_path, driver="GeoJSON")

output_path


  STRATO TEMA CLASSE       ID_ZRIL    FEATURE_ID building_type building_use  \
0     02   01     02  MI_2006_0033  000000000001          0101         0291   
1     02   01     02  MI_2006_0033  000000000002          0101         0201   
2     02   01     02  MI_2006_0033  000000000003          0101         0201   
3     02   01     02  MI_2006_0033  000000000004          0101         0201   
4     02   01     02  MI_2006_0033  000000000005          0101         0201   

  building_status   building_id EDIFC_CASS  \
0            0401  E00000000001  000000001   
1            0403  E00000000002  000000002   
2            0403  E00000000003  000000003   
3            0403  E00000000004  000000004   
4            0403  E00000000005  000000005   

                                            geometry  
0  POLYGON Z ((9.22342 45.49639 124.3394, 9.22342...  
1  POLYGON Z ((9.23502 45.49384 123.3962, 9.23492...  
2  POLYGON Z ((9.22071 45.49534 124.1928, 9.22097...  
3  POLYGON Z ((9.22097 45.49

'data/Buildings/buildings_clean_full.geojson'

In [6]:
invalid_mask = ~buildings.geometry.is_valid
invalid_count = invalid_mask.sum()
invalid_count


np.int64(20)

In [8]:
buildings.geometry = buildings.geometry.buffer(0)

In [9]:
from shapely.geometry import Polygon, MultiPolygon

buildings = buildings[
    buildings.geometry.apply(lambda geom: isinstance(geom, (Polygon, MultiPolygon)))
]


In [10]:
output_path = r"data/Buildings/buildings_clean_validated.geojson"
buildings.to_file(output_path, driver="GeoJSON")

output_path


'data/Buildings/buildings_clean_validated.geojson'

In [11]:
# 1) Reproject to metric CRS (EPSG:6707) for area calculation
buildings_utm = buildings.to_crs(epsg=6707)

# 2) Compute footprint area in square meters
buildings_utm["area_m2"] = buildings_utm.geometry.area

# 3) Reproject back to WGS84 (EPSG:4326) while keeping the area column
buildings = buildings_utm.to_crs(epsg=4326)

# Check result
buildings[["building_id", "area_m2"]].head()


,building_id,area_m2
0,E00000000001,727.719472
1,E00000000002,270.704927
2,E00000000003,281.929276
3,E00000000004,347.607947
4,E00000000005,191.138372


In [12]:
# 4) Compute centroid of each building polygon
centroids = buildings.geometry.centroid

# 5) Extract latitude and longitude of centroid
buildings["centroid_lon"] = centroids.x
buildings["centroid_lat"] = centroids.y

# Quick check
buildings[["building_id", "centroid_lon", "centroid_lat"]].head()


C:\Users\Nima\AppData\Local\Temp\ipykernel_7612\384410514.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids = buildings.geometry.centroid


,building_id,centroid_lon,centroid_lat
0,E00000000001,9.223222,45.496484
1,E00000000002,9.234857,45.493874
2,E00000000003,9.220783,45.495266
3,E00000000004,9.220984,45.495336
4,E00000000005,9.220669,45.495458


In [13]:
output_path = r"data/Buildings/buildings_geom_features.geojson"
buildings.to_file(output_path, driver="GeoJSON")

output_path


'data/Buildings/buildings_geom_features.geojson'

In [1]:
# Load clean buildings + climate GeoJSON (already validated locally)
buildings_path = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\data\Buildings\buildings_with_climate_2d.geojson"
buildings_gdf = gpd.read_file(buildings_path)

# Normalize building address column
buildings_gdf["INDIRIZZO_NORM"] = buildings_gdf["EDIFC_CASS"].astype(str).str.strip().str.upper()


NameError: name 'gpd' is not defined